In [1]:
import json

In [3]:
# Reading a single metadata or pdf_parses file
file_path = "/Work/Experiments/IISER/s2orc-complete/20260529_115541_00034_tz6wk_aa3c8ddf-857f-4e12-9645-8589a7cea0b4"

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        paper_data = json.loads(line)
        # Access data fields like 'title', 'abstract', or 'mag_id'
        # print(paper_data.get('title'))
        break


In [4]:
paper_data.keys()

dict_keys(['corpusid', 'externalids', 'content'])

In [5]:
paper_data["corpusid"]

201803469

In [6]:
raw_content = paper_data["content"]
raw_content

{'source': {'pdfurls': None,
  'pdfsha': '7c2164db181b7857bfa7383f945f6ce52bdb6747',
  'oainfo': {'license': 'CCBY',
   'openaccessurl': 'https://www.frontiersin.org/articles/10.3389/fonc.2019.00829/pdf',
   'status': 'GOLD'}},
 'text': '\nComputed Tomography Radiomic Nomogram for Preoperative Prediction of Extrathyroidal Extension in Papillary Thyroid Carcinoma\nSeptember 2019\n\nBo Gao \nSeyedmehdi Payabvash \nJie Tian jie.tian@ia.ac.cn \nWei Lu \nChen B \nZhong L \nDong D \nFang MZheng J \nYu C \nDai Q \nZhang L \nTian J \nLu W \nJin Y \nBin Chen \nLianzhen Zhong \nDi Dong \nJianjun Zheng \n† \nMengjie Fang \nChunyao Yu \nDepartment of Medical Imaging\nHwa Mei Hospital\nUniversity of Chinese Academy of Sciences\nNingboChina\n\nInstitute of Automation\nCAS Key Laboratory of Molecular Imaging\nChinese Academy of Sciences\nBeijingChina\n\nQi Dai \nDepartment of Medical Imaging\nHwa Mei Hospital\nUniversity of Chinese Academy of Sciences\nNingboChina\n\nInstitute of Automation\nCAS Key 

In [7]:
def extract_info(raw_text:str,annotations:dict,info_type:str):
    annotations = eval(annotations[info_type]) # this can be more than one entries
    attribute_texts = []
    print(f'Total {len(annotations)} markers')
    for item in annotations:
        start = int(item['start'])
        end = int(item["end"])
        data = raw_text[start:end]
        attribute_texts.append(data)
    return attribute_texts

def extract_info_w_attributes(raw_text:str,annotations:dict,info_type:str):
    annotations = eval(annotations[info_type]) # this can be more than one entries
    attribute_texts = []
    print(f'Total {len(annotations)} markers')
    for item in annotations: # here we can have an attribute key as well
        start = int(item['start'])
        end = int(item["end"])
        if 'attributes' in item.keys():
            attribute = item['attributes']
            attribute_vals = []
            for k in attribute.keys():
                at_val = attribute[k]
                attribute_vals.append(at_val)
        else:
            attribute_vals = None
        data = raw_text[start:end]
        attribute_texts.append((attribute_vals,data))
    return attribute_texts

In [9]:
all_text = raw_content["text"]
all_annotations = raw_content["annotations"]
info = extract_info_w_attributes(all_text,all_annotations,"bibentry")
for i in info:
    print(i)

Total 39 markers
(['b0', '10.1016/S0140-6736(16)30172-6'], '. M E Cabanillas, D G Mcfadden, C Durante, 10.1016/S0140-6736(16)30172-6Thyroid cancer. Lancet. 38816Cabanillas ME, Mcfadden DG, Durante C. Thyroid cancer. Lancet. (2016) 388:2783-95. doi: 10.1016/S0140-6736(16)30172-6')
(['1514000', 'b1', '10.3322/caac.21551'], 'Cancer statistics. R L Siegel, K D Miller, A Jemal, 10.3322/caac.21551CA Cancer J Clin. 69Siegel RL, Miller KD, Jemal A. Cancer statistics, 2019. CA Cancer J Clin. (2019) 69:7-34. doi: 10.3322/caac.21551')
(['b2', '10.3322/caac.21338'], 'Cancer statistics in China. W Chen, R Zheng, P D Baade, S Zhang, H Zeng, F Bray, 10.3322/caac.21338Ca Cancer J Clin. 66Chen W, Zheng R, Baade PD, Zhang S, Zeng H, Bray F, et al. Cancer statistics in China, 2015. Ca Cancer J Clin. (2016) 66:115-32. doi: 10.3322/caac.21338')
(['b3'], 'SEER Data Submission, Posted to the SEER Web Site. N Howlader, A Noone, M Krapcho, D Miller, K Bishop, C L Kosary, Cancer Statistics Review. National Canc

In [10]:
# general attributes
all_text = raw_content["text"]
attribute = 'title'
annotation = eval(raw_content["annotations"][attribute]) # this can be more than one entries
all_markers = []
for item in annotation:
    start = int(item['start'])
    end = int(item["end"])
    all_markers.append((start,end))
attribute_text = "" # empty string
for marker in all_markers: # each element is (start, end)
    attribute_text += all_text[marker[0]:marker[1]] + "\n"
attribute_text

'Computed Tomography Radiomic Nomogram for Preoperative Prediction of Extrathyroidal Extension in Papillary Thyroid Carcinoma\nComputed Tomography Radiomic Nomogram for Preoperative Prediction of Extrathyroidal Extension in Papillary Thyroid Carcinoma\n'

In [11]:
def get_regular_attributes(raw_content:dict,attribute:str):
    # general attributes
    all_text = raw_content["text"]
    annotation = eval(raw_content["annotations"][attribute]) # this can be more than one entries
    all_markers = []
    for item in annotation:
        start = int(item['start'])
        end = int(item["end"])
        all_markers.append((start,end))
    attribute_text = "" # empty string
    for marker in all_markers: # each element is (start, end)
        attribute_text += all_text[marker[0]:marker[1]] + "\n"
    return attribute_text

def get_citations(raw_content:dict):
    # only for bibentry
    all_text = raw_content["text"]
    attribute = 'bibentry'
    annotation = eval(raw_content["annotations"][attribute]) # this can be more than one entries
    all_markers = []
    for item in annotation:
        cite_id = item["attributes"]["id"]
        if "matched_paper_id" in item["attributes"].keys():
            cite_paper_id = item["attributes"]["matched_paper_id"]
        else:
            cite_paper_id = -1
        start = int(item['start'])
        end = int(item["end"])
        all_markers.append((start,end,cite_id,cite_paper_id))
    attribute_text = []
    for marker in all_markers: # each element is (start, end)
        data = {"id":marker[2],"paper":all_text[marker[0]:marker[1]],"ss_id":marker[-1]}
        attribute_text.append(data)
    return attribute_text

In [12]:
attribute_text = get_regular_attributes(raw_content,"abstract")
print(attribute_text)

Objectives: Determining the presence of extrathyroidal extension (ETE) is important for patients with papillary thyroid carcinoma (PTC) in selecting the proper surgical approaches. This study aimed to explore a radiomic model for preoperative prediction of ETE in patients with PTC.Methods: The study included 624 PTC patients (without ETE, n = 448; with minimal ETE, n = 52; with gross ETE, n = 124) whom were divided randomly into training (n = 437) and validation (n = 187) cohorts; all data were gathered between January 2016 and November 2017. Radiomic features were extracted from computed tomography (CT) images of PTCs. Key radiomic features were identified and incorporated into a radiomic signature. Combining the radiomic signature with clinical risk factors, a radiomic nomogram was constructed using multivariable logistic regression. Delong test was used to compare different receiver operating characteristic curves.Results:Five key radiomic features were incorporated into the radiomi

In [13]:
# only for bibentry
all_text = raw_content["text"]
attribute = 'bibentry'
annotation = eval(raw_content["annotations"][attribute]) # this can be more than one entries
all_markers = []
for item in annotation:
    cite_id = item["attributes"]["id"]
    if "matched_paper_id" in item["attributes"].keys():
        cite_paper_id = item["attributes"]["matched_paper_id"]
    else:
        cite_paper_id = -1
    start = int(item['start'])
    end = int(item["end"])
    all_markers.append((start,end,cite_id,cite_paper_id))
attribute_text = []
for marker in all_markers: # each element is (start, end)
    data = {"id":marker[2],"paper":all_text[marker[0]:marker[1]],"ss_id":marker[-1]}
    attribute_text.append(data)
for d in attribute_text:
    print(f"[{d["id"]}]\nSS ID:{d["ss_id"]}\n{d["paper"]}\n-------")

[b0]
SS ID:-1
. M E Cabanillas, D G Mcfadden, C Durante, 10.1016/S0140-6736(16)30172-6Thyroid cancer. Lancet. 38816Cabanillas ME, Mcfadden DG, Durante C. Thyroid cancer. Lancet. (2016) 388:2783-95. doi: 10.1016/S0140-6736(16)30172-6
-------
[b1]
SS ID:1514000
Cancer statistics. R L Siegel, K D Miller, A Jemal, 10.3322/caac.21551CA Cancer J Clin. 69Siegel RL, Miller KD, Jemal A. Cancer statistics, 2019. CA Cancer J Clin. (2019) 69:7-34. doi: 10.3322/caac.21551
-------
[b2]
SS ID:-1
Cancer statistics in China. W Chen, R Zheng, P D Baade, S Zhang, H Zeng, F Bray, 10.3322/caac.21338Ca Cancer J Clin. 66Chen W, Zheng R, Baade PD, Zhang S, Zeng H, Bray F, et al. Cancer statistics in China, 2015. Ca Cancer J Clin. (2016) 66:115-32. doi: 10.3322/caac.21338
-------
[b3]
SS ID:-1
SEER Data Submission, Posted to the SEER Web Site. N Howlader, A Noone, M Krapcho, D Miller, K Bishop, C L Kosary, Cancer Statistics Review. National Cancer InstituteHowlader N, Noone A, Krapcho M, Miller D, Bishop K, Ko

'bibref' - ids for citations